[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-08-mlflow-projects.ipynb#scrollTo=1a2b3c4d)

---
# Day 8 · MLflow Projects — Packaging Reproducible Code
**certified-journeys / mlflow-certified** · Practice · Reproducible ML Pipelines

> **Goal for today:** Create an MLproject file with entry points and parameterized hyperparameters, run experiments locally with `mlflow run`, and understand how Projects pin dependencies for reproducible execution.


In [ ]:
%pip install -q mlflow scikit-learn pandas numpy


## Step 1 · What is an MLflow Project?

An **MLflow Project** is a convention for packaging ML code so that *anyone* can reproduce your experiment with a single command:

```bash
mlflow run . -P learning_rate=0.01
```

A project is a directory (or Git repo) that contains:

| Component | Required? | Purpose |
|---|---|---|
| `MLproject` | Yes | Declares name, env, and entry points |
| `conda.yaml` or `requirements.txt` | Yes (one of) | Pins all Python dependencies |
| Entry point script(s) | Yes | The actual training code |
| `.gitignore` | Recommended | Excludes data, checkpoints |

Without an `MLproject` file, MLflow assumes the project has a single entry point: `main` that runs `python main.py`.


In [ ]:
import os
import pathlib
import subprocess
import textwrap
import mlflow
from mlflow.tracking import MlflowClient

# Create a temporary project directory inside the Colab working directory
PROJECT_DIR = pathlib.Path("/tmp/iris_project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print("Files created so far:", list(PROJECT_DIR.iterdir()))


**What just happened?**
- We created `/tmp/iris_project` — this is where we'll write the `MLproject` file, the entry point script, and the conda/pip environment spec.
- Using `/tmp` avoids polluting the Colab working directory and ensures a clean project tree.
- In a real workflow this would be the root of your Git repository.


## Step 2 · Write the `MLproject` File

The `MLproject` file is YAML. Its three top-level keys are:

| Key | Required | Description |
|---|---|---|
| `name` | Yes | Human-readable project name |
| `python_env` / `conda_env` / `docker_env` | Yes (one) | Environment spec file reference |
| `entry_points` | Yes | Named commands with parameters |

Each entry point parameter has a `type` (`float`, `int`, `str`, `path`, `uri`) and an optional `default` value. Parameters not listed cannot be passed at runtime.


In [ ]:
mlproject_content = textwrap.dedent("""\
name: iris-classifier

python_env: python_env.yaml

entry_points:
  train:
    parameters:
      learning_rate:
        type: float
        default: 0.1
      n_estimators:
        type: int
        default: 50
      max_depth:
        type: int
        default: 5
    command: "python train.py --learning_rate {learning_rate} --n_estimators {n_estimators} --max_depth {max_depth}"

  evaluate:
    parameters:
      model_uri:
        type: str
    command: "python evaluate.py --model_uri {model_uri}"
""")

(PROJECT_DIR / "MLproject").write_text(mlproject_content)
print("MLproject file written:")
print(mlproject_content)


**What just happened?**
- **`python_env`** references a `python_env.yaml` file (pip-based; lighter than conda). Use `conda_env` if you need non-Python dependencies.
- The `{learning_rate}` placeholder in the `command` is substituted by MLflow at runtime from the `-P` flags.
- We defined two entry points: `train` (the main one) and `evaluate` — a project can have as many as needed.


## Step 3 · Write the Environment Spec and Training Script

MLflow supports three environment types:

| Type | File | Best for |
|---|---|---|
| `python_env` | `python_env.yaml` | Lightweight pip-only setups |
| `conda_env` | `conda.yaml` | Environments with C/R/Java deps |
| `docker_env` | Dockerfile path | Fully reproducible including OS |

For CI/CD reproducibility, always pin exact package versions in your environment file.


In [ ]:
# Write python_env.yaml — MLflow's pip-based environment spec
python_env_content = textwrap.dedent("""\
python: "3.10"
build_dependencies:
  - pip
  - setuptools
  - wheel
dependencies:
  - mlflow>=2.3.0
  - scikit-learn>=1.2.0
  - pandas>=1.5.0
  - numpy>=1.23.0
""")
(PROJECT_DIR / "python_env.yaml").write_text(python_env_content)

# Write the training entry point script
train_script = textwrap.dedent("""\
import argparse
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--learning_rate", type=float, default=0.1)
    parser.add_argument("--n_estimators",  type=int,   default=50)
    parser.add_argument("--max_depth",     type=int,   default=5)
    args = parser.parse_args()

    iris = load_iris(as_frame=True)
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=42
    )

    with mlflow.start_run():
        mlflow.log_params(vars(args))

        clf = GradientBoostingClassifier(
            learning_rate=args.learning_rate,
            n_estimators=args.n_estimators,
            max_depth=args.max_depth,
            random_state=42,
        )
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)

        mlflow.log_metric("accuracy", accuracy_score(y_test, preds))
        mlflow.log_metric("f1_macro", f1_score(y_test, preds, average="macro"))
        mlflow.sklearn.log_model(clf, artifact_path="model")

        print(f"accuracy={accuracy_score(y_test, preds):.4f}")

if __name__ == "__main__":
    main()
""")
(PROJECT_DIR / "train.py").write_text(train_script)

print("Files written:")
for f in sorted(PROJECT_DIR.iterdir()):
    print(f"  {f.name}")


**What just happened?**
- **`python_env.yaml`** pins exact minimum versions — MLflow creates a virtualenv from this before running the entry point.
- The training script uses **`argparse`** to accept the parameters declared in `MLproject` — it must match the `command` template exactly.
- **`mlflow.start_run()`** inside the script creates a child run when called from `mlflow run` (which sets `MLFLOW_RUN_ID` env var automatically).


## Step 4 · Run the Project Locally with `mlflow run`

The `mlflow run` CLI command:

```bash
mlflow run <project-uri> -e <entry-point> -P <param>=<value> --experiment-name <name>
```

Key flags:

| Flag | Default | Description |
|---|---|---|
| `-e` | `main` | Which entry point to run |
| `-P key=val` | declared defaults | Override a parameter |
| `--experiment-name` | `Default` | Experiment to log runs into |
| `--env-manager` | `local` / `virtualenv` | `local` skips env creation (use in Colab) |
| `--no-conda` | — | Deprecated alias for `--env-manager=local` |

> **Colab note:** We pass `--env-manager=local` to skip virtualenv creation — MLflow uses the already-installed packages.


In [ ]:
import subprocess

# Set up a local tracking URI so runs appear in our SQLite DB
tracking_uri = "sqlite:////tmp/mlflow_projects_demo.db"
experiment_name = "day-08-mlflow-projects"

# Run the 'train' entry point with custom hyperparameters
result = subprocess.run(
    [
        "mlflow", "run",
        str(PROJECT_DIR),                        # project directory
        "-e", "train",                            # entry point
        "-P", "learning_rate=0.05",
        "-P", "n_estimators=80",
        "-P", "max_depth=3",
        "--experiment-name", experiment_name,
        "--env-manager", "local",                 # use current Python environment
    ],
    capture_output=True,
    text=True,
    env={**os.environ, "MLFLOW_TRACKING_URI": tracking_uri},
)

print("STDOUT:", result.stdout[-800:] if len(result.stdout) > 800 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-600:])
print("Return code:", result.returncode)


**What just happened?**
- **`mlflow run <dir>`** read `MLproject`, resolved the `train` entry point, substituted `-P` values into the command template, and executed `python train.py --learning_rate 0.05 …`.
- The `MLFLOW_TRACKING_URI` environment variable overrides any tracking URI set inside the script — this is how you point runs from a project to your central server.
- **`--env-manager=local`** means MLflow does not create a new virtualenv — useful in Colab where packages are already installed.


## Step 5 · Run Multiple Parameter Combinations (Grid Search via Projects)

Because `mlflow run` is a CLI command, you can script a hyperparameter sweep without a framework like Optuna — just loop in Python and spawn runs. Each call creates an independent MLflow run, making comparison easy in the UI.


In [ ]:
import itertools

param_grid = {
    "learning_rate": [0.01, 0.1],
    "n_estimators":  [50, 100],
}

run_results = []

for lr, ne in itertools.product(param_grid["learning_rate"], param_grid["n_estimators"]):
    r = subprocess.run(
        [
            "mlflow", "run", str(PROJECT_DIR),
            "-e", "train",
            "-P", f"learning_rate={lr}",
            "-P", f"n_estimators={ne}",
            "--experiment-name", experiment_name,
            "--env-manager", "local",
        ],
        capture_output=True,
        text=True,
        env={**os.environ, "MLFLOW_TRACKING_URI": tracking_uri},
    )
    status = "OK" if r.returncode == 0 else "FAIL"
    run_results.append({"lr": lr, "ne": ne, "status": status})
    print(f"  lr={lr}, n_estimators={ne} → {status}")

print(f"\nCompleted {len(run_results)} runs")


**What just happened?**
- **`itertools.product`** generates all combinations of hyperparameters — a simple grid search without a heavy framework.
- Each `mlflow run` call is fully isolated: separate run_id, separate metrics, separate artifacts.
- You can now compare all runs in the MLflow UI (`mlflow ui --backend-store-uri sqlite:///…`) and filter/sort by metric.


## Step 6 · Inspect Runs Logged by the Project

After running experiments via `mlflow run`, you use the standard `MlflowClient` or `mlflow.search_runs` API to retrieve and compare results — the same way you would for runs logged directly in a notebook.


In [ ]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri(tracking_uri)

# Search all runs in our experiment
runs_df = mlflow.search_runs(
    experiment_names=[experiment_name],
    order_by=["metrics.accuracy DESC"],
)

# Select the columns we care about
cols = ["run_id", "params.learning_rate", "params.n_estimators",
        "metrics.accuracy", "metrics.f1_macro", "status"]
available = [c for c in cols if c in runs_df.columns]
print(runs_df[available].to_string(index=False))

# Find the best run
if "metrics.accuracy" in runs_df.columns and not runs_df.empty:
    best = runs_df.loc[runs_df["metrics.accuracy"].idxmax()]
    print(f"\nBest run: {best['run_id'][:8]}… accuracy={best['metrics.accuracy']:.4f}")


**What just happened?**
- **`mlflow.search_runs`** returns a pandas DataFrame — you can sort, filter, and visualize results just like any DataFrame.
- **`order_by=["metrics.accuracy DESC"]`** brings the best run to the top — equivalent to the UI's column sort.
- In a real pipeline, you'd take the `run_id` of the best run and feed it into `mlflow.register_model` (Day 7 pattern).


## Step 7 · Run a Project from a GitHub URL

MLflow Projects can be run directly from a public GitHub repo — no local checkout needed:

```bash
mlflow run https://github.com/mlflow/mlflow-example -P alpha=0.5
```

You can also pin to a specific Git ref:

```bash
mlflow run https://github.com/mlflow/mlflow-example#v1.0 -P alpha=0.5
```

This is the reproducibility payoff: a paper author can publish a GitHub URL and readers can reproduce any result with a single command.

The cell below shows the command **without running it** to avoid GitHub rate limits in Colab — the syntax is identical to running a local directory.


In [ ]:
# Demonstrates the GitHub URL run syntax — shown without executing to avoid
# network dependencies in Colab. In a real environment, remove the dry_run flag.
github_command = [
    "mlflow", "run",
    "https://github.com/mlflow/mlflow-example",
    "-P", "alpha=0.42",
    "-P", "l1_ratio=0.1",
    "--experiment-name", "github-example",
    "--env-manager", "local",
]

print("Command that would run the official MLflow example project from GitHub:")
print(" ".join(github_command))
print()
print("To actually run it (requires network access and ~30s):")
print("  Uncomment the subprocess.run(...) call below")
print()

# Uncomment to run:
# result = subprocess.run(github_command, capture_output=True, text=True,
#                         env={**os.environ, "MLFLOW_TRACKING_URI": tracking_uri})
# print(result.stdout[-600:])

# Show the project structure expected by mlflow-example
print("The mlflow/mlflow-example repo contains:")
print("  MLproject          ← entry points for train")
print("  conda.yaml         ← conda environment spec")
print("  train.py           ← ElasticNet regression on wine quality data")


**What just happened?**
- The **GitHub URL format** (`https://github.com/…`) is the exact same interface as the local directory path — MLflow clones the repo and looks for `MLproject`.
- Appending `#<git-ref>` (branch, tag, or commit SHA) pins the exact code version — critical for reproducibility.
- The mlflow-example project uses `conda.yaml`; our project uses `python_env.yaml` — both are valid and MLflow handles them transparently.


## Step 8 · Understanding Dependency Pinning

MLflow Projects support three environment managers. Understanding trade-offs helps you choose the right one:

| Manager | How it pins | Reproducibility | Speed |
|---|---|---|---|
| `local` | None — uses current env | Low (environment not controlled) | Fast |
| `virtualenv` (python_env) | pip requirements | High for Python packages | Medium |
| `conda` (conda_env) | conda + pip | Highest (includes native libs) | Slow |
| `docker` (docker_env) | Full container image | Complete (OS + Python + libs) | Depends |

For production ML pipelines, use conda or docker. For quick sharing with colleagues on similar setups, virtualenv is sufficient.


In [ ]:
# Show an equivalent conda.yaml for the same project
conda_yaml_equivalent = textwrap.dedent("""\
name: iris-classifier-env
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10.0
  - pip=23.0
  - pip:
    - mlflow>=2.3.0
    - scikit-learn>=1.2.0
    - pandas>=1.5.0
    - numpy>=1.23.0
""")

print("Equivalent conda.yaml (for conda_env field in MLproject):")
print(conda_yaml_equivalent)

# Show how to auto-generate a conda.yaml from the current environment
print("To auto-generate from your current conda env:")
print("  conda env export > conda.yaml")
print()
print("To auto-generate a requirements.txt from current pip env:")
print("  pip freeze > requirements.txt")
print("  # Then reference it in MLproject as: python_env: requirements.txt")

# Programmatic way to capture current env versions
import importlib.metadata
key_packages = ["mlflow", "scikit-learn", "pandas", "numpy"]
print("\nCurrent versions (for pinning):")
for pkg in key_packages:
    try:
        ver = importlib.metadata.version(pkg)
        print(f"  {pkg}=={ver}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg} not installed")


**What just happened?**
- A `conda.yaml` using the `conda-forge` channel is more portable across operating systems than pip alone — conda resolves native binary dependencies.
- **`conda env export`** captures exact versions including build strings — the most reproducible option but can fail on different OS/CPU architectures.
- **`importlib.metadata.version`** is the modern (Python 3.8+) way to read a package's installed version without importing it.


In [ ]:
# Challenge: Add a second entry point
#
# Task: Write a new entry point script 'evaluate.py' that:
#   1. Accepts --model_uri (str) as a CLI argument
#   2. Loads the model with mlflow.sklearn.load_model(model_uri)
#   3. Runs inference on the Iris test set
#   4. Logs accuracy and a classification_report as an artifact with mlflow.log_text()
#
# The MLproject already declares the 'evaluate' entry point (see Step 2).
#
# Hints:
#   - Use argparse as in train.py
#   - sklearn.metrics.classification_report(y_test, preds) returns a string
#   - mlflow.log_text(text, artifact_file="classification_report.txt")
#   - Remember to wrap everything in with mlflow.start_run():

evaluate_script = """
import argparse
import mlflow
# Your solution here — write the evaluate.py script
"""

# Uncomment to write your solution:
# (PROJECT_DIR / "evaluate.py").write_text(evaluate_script)
# Then run it:
# subprocess.run(["mlflow", "run", str(PROJECT_DIR), "-e", "evaluate",
#                 "-P", f"model_uri=runs:/<run_id>/model",
#                 "--env-manager", "local"], ...)


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| `MLproject` file | YAML with `name`, `python_env`/`conda_env`, and `entry_points` |
| Entry point parameters | Typed (`float`, `int`, `str`) with defaults; passed via `-P key=val` |
| `mlflow run .` | Runs the default `main` entry point from the current directory |
| `-e <name>` flag | Selects a non-default entry point |
| GitHub URL | `mlflow run https://github.com/…#<ref>` — reproducible from any machine |
| `--env-manager` | `local` (fast, no isolation), `virtualenv` (pip), `conda` (full) |
| `python_env.yaml` | Pip-based env spec: python version + build deps + dependencies |

> **Tip:** An MLproject file turns any directory into a runnable, shareable experiment — teammates can reproduce your results with a single `mlflow run` command.

---
## What's next
**Day 9** → Custom Python Function (PyFunc) models — wrap any model class with preprocessing into a universal MLflow model format that supports serving, signature validation, and registry support.

Mark Day 8 complete in your [tracker](../index.html).
